# DSAR × Lakeflow · 03 · Validation (click-to-run, one query per cell)

**Read-only** checks — set the widgets and **Run all**. Nothing is modified. Each
check is its own `%sql` cell so you get a rendered table per result. Widgets are
substituted via `${catalog}` / `${schema}` / `${volume}`.

Point the widgets at the variant you ran (append = `allegiant_air_sdp_dsar`,
CDC = `allegiant_air_sdp_dsar_cdc`).


## 0. Widgets (run this first)


In [ ]:
dbutils.widgets.removeAll()

In [ ]:
dbutils.widgets.text("catalog", "dkushari_uc", "1 Catalog")
dbutils.widgets.text("schema", "allegiant_air_sdp_dsar", "2 Schema")
dbutils.widgets.text("volume", "raw_user", "3 Landing volume")
print("catalog =", dbutils.widgets.get("catalog"),
      "| schema =", dbutils.widgets.get("schema"),
      "| volume =", dbutils.widgets.get("volume"))


### 1. Landing file records — INITIAL (expect ~10,000)


In [ ]:
%sql
SELECT count(*) AS initial_file_records
FROM read_files('/Volumes/${catalog}/${schema}/${volume}/landing/initial',
                format => 'json', recursiveFileLookup => 'true');


### 2. Landing file records — INCREMENTAL (0 until 00b runs; +24 after)


In [ ]:
%sql
SELECT count(*) AS incremental_file_records
FROM read_files('/Volumes/${catalog}/${schema}/${volume}/landing/incremental',
                format => 'json', recursiveFileLookup => 'true');


### 3. Landing file records — ALL landing/ (recursive total)


In [ ]:
%sql
SELECT count(*) AS all_landing_records
FROM read_files('/Volumes/${catalog}/${schema}/${volume}/landing',
                format => 'json', recursiveFileLookup => 'true');


### 4. Medallion table counts (raw → bronze → silver → gold)


In [ ]:
%sql
SELECT 'raw_user'    AS table, count(*) AS rows FROM ${catalog}.${schema}.raw_user
UNION ALL SELECT 'bronze_user', count(*) FROM ${catalog}.${schema}.bronze_user
UNION ALL SELECT 'silver_user', count(*) FROM ${catalog}.${schema}.silver_user
UNION ALL SELECT 'gold_user',   count(*) FROM ${catalog}.${schema}.gold_user
ORDER BY table;


### 5. PII masking — silver sample (email/full_name should be ***REDACTED***, in-JSON contact.* redacted)


In [ ]:
%sql
SELECT user_id, email, full_name, revenue, profile_json
FROM ${catalog}.${schema}.silver_user
LIMIT 5;


### 6. PII masking — silver rows with UN-redacted email (expect 0)


In [ ]:
%sql
SELECT count(*) AS unredacted_email_rows
FROM ${catalog}.${schema}.silver_user
WHERE email <> '***REDACTED***';


### 7. DSAR queue — full table


In [ ]:
%sql
SELECT * FROM ${catalog}.${schema}.dsar_request ORDER BY request_id;


### 8. DSAR queue — by status & type


In [ ]:
%sql
SELECT status, request_type, count(*) AS n
FROM ${catalog}.${schema}.dsar_request
GROUP BY status, request_type
ORDER BY status, request_type;


### 9. No-trace check — cleartext email per requested subject, per layer (run after `02`)

For a **COMPLETE** request, every count below should be **0** (DELETE subjects gone;
OBFUSCATE subjects have their cleartext email redacted). `FILE_records` reads the
volume landing files. `gold_user` has no email column, so it's omitted here.


In [ ]:
%sql
WITH subj AS (
  SELECT request_id, lower(subject_email) AS email, request_type, status
  FROM ${catalog}.${schema}.dsar_request
),
files AS (
  SELECT lower(email) AS email
  FROM read_files('/Volumes/${catalog}/${schema}/${volume}/landing',
                  format => 'json', recursiveFileLookup => 'true')
)
SELECT
  s.request_id, s.request_type, s.status,
  (SELECT count(*) FROM ${catalog}.${schema}.raw_user    r WHERE lower(r.email)=s.email) AS raw_clear,
  (SELECT count(*) FROM ${catalog}.${schema}.bronze_user b WHERE lower(b.email)=s.email) AS bronze_clear,
  (SELECT count(*) FROM ${catalog}.${schema}.silver_user v WHERE lower(v.email)=s.email) AS silver_clear,
  (SELECT count(*) FROM files f WHERE f.email=s.email)                                    AS file_clear
FROM subj s
ORDER BY s.request_id;


### 10. Summary snapshot


In [ ]:
%sql
SELECT
  (SELECT count(*) FROM ${catalog}.${schema}.raw_user)                              AS raw_rows,
  (SELECT count(*) FROM ${catalog}.${schema}.gold_user)                             AS gold_customers,
  (SELECT count(*) FROM ${catalog}.${schema}.dsar_request WHERE status='PENDING')   AS dsar_pending,
  (SELECT count(*) FROM ${catalog}.${schema}.dsar_request WHERE status='COMPLETE')  AS dsar_complete;
